# Etapa 3 — Identidad del dinosaurio con Ollama (notebook SageMaker `ollama`, ml.m5.xlarge)

Ollama corre en Docker dentro de esta instancia (lo levanta el lifecycle script `on-start.sh`). Ninguna solicitud sale a un proveedor externo.

In [ ]:
import subprocess
def sh(cmd): print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout)
sh("docker info --format 'data-root de Docker: {{.DockerRootDir}}'")
sh("docker ps --format 'table {{.Names}}\t{{.Image}}\t{{.Status}}\t{{.Ports}}'")
sh('df -h /home/ec2-user/SageMaker | tail -1')

## Modelo descargado: nombre y versión

In [2]:
import requests, json
OLLAMA = 'http://localhost:11434'
print('Ollama', requests.get(f'{OLLAMA}/api/version').json()['version'])
for m in requests.get(f'{OLLAMA}/api/tags').json()['models']:
    d = m['details']
    print(f"{m['name']:15} digest {m['digest'][:12]}  {d['parameter_size']}  {d['quantization_level']}  {m['size']/1e9:.1f} GB")

Ollama 0.40.0
gemma4:e2b      digest b37049369adf  4.6B  Q4_K_M  4.6 GB


## Prueba: el modelo recibe una instrucción y responde

In [3]:
r = requests.post(f'{OLLAMA}/api/generate', json={'model': 'gemma4:e2b', 'prompt': 'Responde en una frase: ¿qué es un dinosaurio?', 'stream': False}).json()
print(r['response'])
print(f"{r['eval_count']} tokens en {r['total_duration']/1e9:.1f} s")

Un dinosaurio es un tipo de reptil extinto que dominó la Tierra durante la era Mesozoica.
243 tokens en 28.5 s


## Identidad del dinosaurio seleccionado en la etapa 2
Se le dan al modelo ejemplos de raíces reales (-saurus, -raptor, -odon, -venator…) y se pide un JSON con apariencia, hábitat, alimentación, comportamiento y rasgo distintivo. Es el mismo prompt que usa la app (`4_app/lambda/lambda_function.py`).

In [4]:
NOMBRE = 'Limedsaurus'
PROMPT = open('/home/ec2-user/SageMaker/lab-dinos/prompt_identidad.txt').read().format(nombre=NOMBRE)
r = requests.post(f'{OLLAMA}/api/chat', json={
    'model': 'gemma4:e2b', 'stream': False, 'think': False, 'format': 'json', 'options': {'temperature': 0.8},
    'messages': [{'role': 'user', 'content': PROMPT}]}).json()
identidad = json.loads(r['message']['content'])
for k, v in identidad.items():
    print(f'{k:20} {v}')
json.dump({'nombre': NOMBRE, **identidad}, open(f'/home/ec2-user/SageMaker/lab-dinos/identidad_{NOMBRE.lower()}.json', 'w'), ensure_ascii=False, indent=1)

significado_nombre   Lagarto de las Limas
periodo              Jurásico Tardío
tamano               Largo de 12 metros, peso estimado de 5 toneladas
apariencia           Piel escamosa de color verde oliva con placas óseas prominentes a lo largo del lomo.
habitat              Selvas húmedas y zonas pantanosas del este de la isla.
alimentacion         Herbívoro, dieta basada en helechos y plantas acuáticas.
comportamiento       Lento y sigiloso, se mueve a través de la densa vegetación.
rasgo_distintivo     Una cresta ósea prominente en la parte posterior de la cabeza, similar a un lophus.


> **Nota.** Esta celda muestra que Ollama genera la identidad desde el notebook. Como usa temperatura 0.8, cada ejecución da una identidad distinta (esta salió herbívora). La identidad **conservada** y usada en las etapas siguientes (imagen y chat) es la que generó la app con este mismo prompt y modelo: `identidad_limedsaurus.json` (id `limedsaurus-10553b`, carnívoro, gris pizarra con placas óseas y cresta).